# Training Dataset EDA

Exploratory data analysis of `site_training_data.parquet` — 26K active sites with 12+ months history.

Covers:
1. Shape & missing values
2. Target distribution (`avg_monthly_revenue`, `avg_daily_revenue`)
3. Numeric feature distributions
4. Categorical breakdowns
5. Correlation heatmap
6. Geographic distribution
7. Boolean flag prevalence

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 5)

df = pl.read_parquet("../data/processed/site_training_data.parquet")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

## 1. Missing Values & Basic Stats

In [ ]:
# Null counts — only show columns with any nulls
null_counts = df.null_count().unpivot()
nulls = null_counts.filter(pl.col("value") > 0).sort("value", descending=True)
if len(nulls) == 0:
    print("No null values in any column.")
else:
    print(f"{len(nulls)} columns with nulls:")
    for row in nulls.iter_rows(named=True):
        pct = row["value"] / len(df) * 100
        print(f"  {row['variable']:45s} {row['value']:>6,} ({pct:.1f}%)")

In [ ]:
# Numeric summary for key columns
key_numeric = [
    "avg_monthly_revenue", "avg_daily_revenue", "total_revenue",
    "active_months", "total_active_days", "screen_count",
    "avg_household_income", "median_age",
    "min_distance_to_nearest_site_mi", "min_distance_to_interstate_mi",
]
available = [c for c in key_numeric if c in df.columns]
df.select(available).describe()

## 2. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

rev = df["avg_monthly_revenue"].drop_nulls().to_numpy()

# Raw distribution
axes[0].hist(rev, bins=80, edgecolor="white", linewidth=0.3)
axes[0].set_title("avg_monthly_revenue")
axes[0].set_xlabel("Revenue ($)")
axes[0].axvline(np.median(rev), color="red", ls="--", label=f"median={np.median(rev):.0f}")
axes[0].legend()

# Log-transformed
log_rev = df["log_total_revenue"].drop_nulls().to_numpy()
axes[1].hist(log_rev, bins=80, edgecolor="white", linewidth=0.3, color="#e07b54")
axes[1].set_title("log(total_revenue)")
axes[1].set_xlabel("Log revenue")

# Daily revenue
daily = df["avg_daily_revenue"].drop_nulls().to_numpy()
axes[2].hist(daily, bins=80, edgecolor="white", linewidth=0.3, color="#54a0e0")
axes[2].set_title("avg_daily_revenue")
axes[2].set_xlabel("Revenue ($/day)")
axes[2].axvline(np.median(daily), color="red", ls="--", label=f"median={np.median(daily):.1f}")
axes[2].legend()

fig.suptitle("Revenue Target Distributions", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Revenue percentiles — useful for lookalike threshold selection
percentiles = [5, 10, 20, 25, 50, 75, 80, 90, 95, 99]
rev_series = df["avg_monthly_revenue"].drop_nulls().to_numpy()
print("avg_monthly_revenue percentiles:")
for p in percentiles:
    val = np.percentile(rev_series, p)
    print(f"  p{p:>2d}: ${val:>10,.2f}")

## 3. Numeric Feature Distributions

In [ ]:
# Histograms for key numeric features
numeric_features = [
    "active_months", "total_active_days", "screen_count",
    "avg_household_income", "median_age",
    "min_distance_to_nearest_site_mi", "min_distance_to_interstate_mi",
    "min_distance_to_kroger_mi", "min_distance_to_walmart_mi",
]
available = [c for c in numeric_features if c in df.columns]
ncols = 3
nrows = (len(available) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
axes = axes.flatten()

for i, col in enumerate(available):
    vals = df[col].drop_nulls().to_numpy()
    axes[i].hist(vals, bins=50, edgecolor="white", linewidth=0.3)
    axes[i].set_title(col, fontsize=10)
    axes[i].axvline(np.median(vals), color="red", ls="--", alpha=0.7)

# Hide unused subplots
for j in range(len(available), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Numeric Feature Distributions (red = median)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Categorical Breakdowns

In [ ]:
cat_cols = ["network", "program", "experience_type", "hardware_type", "retailer"]
available = [c for c in cat_cols if c in df.columns]

fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 5))
if len(available) == 1:
    axes = [axes]

for i, col in enumerate(available):
    counts = df.group_by(col).len().sort("len", descending=True)
    top = counts.head(15)  # Show top 15 categories
    labels = top[col].to_list()
    values = top["len"].to_list()

    axes[i].barh(range(len(labels)), values)
    axes[i].set_yticks(range(len(labels)))
    axes[i].set_yticklabels([str(l)[:20] for l in labels], fontsize=8)
    axes[i].set_title(col, fontsize=11)
    axes[i].invert_yaxis()
    axes[i].set_xlabel("Count")

plt.suptitle("Top Categories (training set)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Revenue by network — box plot for top networks
top_networks = (
    df.group_by("network").len()
    .sort("len", descending=True)
    .head(10)["network"].to_list()
)

pdf = df.filter(pl.col("network").is_in(top_networks)).to_pandas()

fig, ax = plt.subplots(figsize=(14, 5))
order = pdf.groupby("network")["avg_monthly_revenue"].median().sort_values(ascending=False).index
sns.boxplot(data=pdf, x="network", y="avg_monthly_revenue", order=order, ax=ax, showfliers=False)
ax.set_title("avg_monthly_revenue by Network (top 10, outliers hidden)")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
# Correlations among key numeric columns
corr_cols = [
    "avg_monthly_revenue", "avg_daily_revenue", "total_revenue",
    "active_months", "total_active_days", "screen_count",
    "avg_household_income", "median_age",
    "avg_monthly_monthly_impressions", "avg_monthly_monthly_nvis",
    "min_distance_to_nearest_site_mi", "min_distance_to_interstate_mi",
    "min_distance_to_kroger_mi", "min_distance_to_walmart_mi",
    "rs_Revenue_95_185", "rs_Revenue_185_370",
]
available = [c for c in corr_cols if c in df.columns]
corr_pdf = df.select(available).to_pandas()
corr = corr_pdf.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, ax=ax,
    xticklabels=[c.replace("avg_monthly_", "").replace("min_distance_to_", "dist_") for c in available],
    yticklabels=[c.replace("avg_monthly_", "").replace("min_distance_to_", "dist_") for c in available],
)
ax.set_title("Correlation Heatmap (key numeric features)", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Geographic Distribution

In [ ]:
# Site locations colored by revenue
geo = df.select(["latitude", "longitude", "avg_monthly_revenue"]).drop_nulls().to_pandas()

fig, ax = plt.subplots(figsize=(14, 8))
sc = ax.scatter(
    geo["longitude"], geo["latitude"],
    c=np.clip(geo["avg_monthly_revenue"], 0, geo["avg_monthly_revenue"].quantile(0.95)),
    cmap="YlOrRd", s=3, alpha=0.6,
)
plt.colorbar(sc, label="avg_monthly_revenue (clipped at p95)", shrink=0.7)
ax.set_title(f"Training Sites ({len(geo):,}) Colored by Revenue")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(-130, -65)
ax.set_ylim(24, 50)
ax.set_aspect(1.3)
plt.tight_layout()
plt.show()

In [ ]:
# Site density by DMA rank
dma = df.select(["dma_rank", "avg_monthly_revenue"]).drop_nulls()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(dma["dma_rank"].to_numpy(), bins=50, edgecolor="white", linewidth=0.3)
axes[0].set_title("Site Count by DMA Rank")
axes[0].set_xlabel("DMA Rank (1 = largest market)")
axes[0].set_ylabel("Site count")

# Revenue vs DMA rank (binned)
dma_pdf = dma.to_pandas()
dma_pdf["dma_bin"] = pd.cut(dma_pdf["dma_rank"], bins=20)
grouped = dma_pdf.groupby("dma_bin", observed=True)["avg_monthly_revenue"].median().reset_index()
axes[1].bar(range(len(grouped)), grouped["avg_monthly_revenue"])
axes[1].set_xticks(range(len(grouped)))
axes[1].set_xticklabels([str(b)[:8] for b in grouped["dma_bin"]], rotation=45, ha="right", fontsize=7)
axes[1].set_title("Median Revenue by DMA Rank Bin")
axes[1].set_ylabel("Median avg_monthly_revenue ($)")

plt.tight_layout()
plt.show()

## 7. Boolean Flag Prevalence

In [ ]:
# Capability and restriction flag prevalence
encoded_cols = sorted([c for c in df.columns if c.endswith("_encoded")])

flag_pcts = []
for col in encoded_cols:
    pct = df[col].sum() / len(df) * 100
    label = col.replace("_encoded", "")
    flag_pcts.append((label, pct))

flag_pcts.sort(key=lambda x: x[1], reverse=True)
labels, pcts = zip(*flag_pcts)

fig, ax = plt.subplots(figsize=(10, 9))
colors = ["#4c78a8" if l.startswith("c_") or l.startswith("schedule") or l.startswith("sellable") else "#e45756" for l in labels]
ax.barh(range(len(labels)), pcts, color=colors)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("% of training sites")
ax.set_title("Boolean Flag Prevalence (blue=capability, red=restriction)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Revenue vs Key Features (scatter)

In [ ]:
scatter_features = [
    "avg_household_income", "active_months", "screen_count",
    "min_distance_to_interstate_mi", "total_active_days", "median_age",
]
available = [c for c in scatter_features if c in df.columns]
ncols = 3
nrows = (len(available) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5 * nrows))
axes = axes.flatten()

for i, col in enumerate(available):
    subset = df.select([col, "avg_monthly_revenue"]).drop_nulls().to_pandas()
    axes[i].scatter(subset[col], subset["avg_monthly_revenue"], s=2, alpha=0.3)
    axes[i].set_xlabel(col, fontsize=9)
    axes[i].set_ylabel("avg_monthly_revenue")
    axes[i].set_title(f"Revenue vs {col}", fontsize=10)
    # Clip y-axis to p99 for readability
    axes[i].set_ylim(0, subset["avg_monthly_revenue"].quantile(0.99))

for j in range(len(available), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Revenue vs Key Features", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 9. Relative Strength Features

In [ ]:
rs_cols = sorted([c for c in df.columns if c.startswith("rs_")])
ncols = 3
nrows = (len(rs_cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3 * nrows))
axes = axes.flatten()

for i, col in enumerate(rs_cols):
    vals = df[col].drop_nulls().to_numpy()
    axes[i].hist(vals, bins=50, edgecolor="white", linewidth=0.3, color="#54a0e0")
    axes[i].axvline(1.0, color="red", ls="--", alpha=0.7, label="neutral (1.0)")
    axes[i].set_title(col.replace("rs_", ""), fontsize=9)
    axes[i].legend(fontsize=7)

for j in range(len(rs_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Relative Strength Distributions (>1 = trending up)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()